# EduLLM1 — Refusal-aware QLoRA fine-tune (point-set topology)

Colab notebook for the selective-prediction SLM (see `PRD.md`).

**Order of operations:**
1. Runtime -> Change runtime type -> **T4 GPU** (free) or **L4** (Pro).
2. Run cells top to bottom.
3. Set `DEV=True` first to smoke-test on **Qwen2.5-3B** (~1h on T4). Then `DEV=False` for **Qwen2.5-Math-7B**.

**Inputs expected in Drive** (`MyDrive/EduLLM1/data/`): `train_chat.jsonl`, `dataset_test.jsonl`.
Outputs (adapter + checkpoints) are written back to Drive so a disconnect doesn't lose progress (PRD risk #14).

In [ ]:
# 1. Check the GPU you actually got.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install deps. Unsloth pulls compatible transformers/peft/trl/bitsandbytes.
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q wandb pyyaml

In [ ]:
# 3. Mount Drive for data in + checkpoints out.
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/EduLLM1'
DATA_DIR = f'{PROJ}/data'
OUT_DIR  = f'{PROJ}/out'
CKPT_DIR = f'{PROJ}/checkpoints'
for d in (DATA_DIR, OUT_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print('Data dir contents:', os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else 'MISSING - upload train_chat.jsonl!')

In [ ]:
# 4. Config. Flip DEV to switch between the fast 3B dev loop and the 7B target.
DEV = True   # <-- True: Qwen2.5-3B smoke test.  False: Qwen2.5-Math-7B.

import torch
IS_BF16 = torch.cuda.is_bf16_supported()  # True on L4/A100, False on T4

if DEV:
    MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
else:
    MODEL_NAME = 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit'

MAX_SEQ_LEN   = 1024          # raise to 2048 on L4 if proofs get truncated
LORA_R        = 16
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
LR            = 2e-4 if IS_BF16 else 1e-4   # lower LR on T4 fp16 (risk #13)
EPOCHS        = 3
BATCH         = 2 if DEV else 1
GRAD_ACCUM    = 4
RUN_NAME      = ('qwen3b-dev' if DEV else 'qwen7b-math')
print(f'Model={MODEL_NAME}  bf16={IS_BF16}  lr={LR}  seq={MAX_SEQ_LEN}')

In [ ]:
# 5. Load base model in 4-bit with Unsloth (M0: base model loads).
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    dtype = None,   # auto: bf16 on L4, fp16 on T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',   # required for memory (PRD §5.2)
    random_state = 13,
)
print('LoRA attached.')

In [ ]:
# 6. Load the chat-format dataset and render to text with the chat template.
from datasets import load_dataset

ds = load_dataset('json', data_files=f'{DATA_DIR}/train_chat.jsonl', split='train')
print('Train examples:', len(ds))

def to_text(ex):
    return {'text': tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False)}

ds = ds.map(to_text)
print(ds[0]['text'][:600])

In [ ]:
# 7. Trainer. Checkpoints to Drive; resume_from_checkpoint survives disconnects.
from trl import SFTTrainer
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir = f'{CKPT_DIR}/{RUN_NAME}',
    per_device_train_batch_size = BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_ratio = 0.03,
    num_train_epochs = EPOCHS,
    learning_rate = LR,
    fp16 = not IS_BF16,
    bf16 = IS_BF16,
    max_grad_norm = 0.3,           # clip — T4 fp16 stability (risk #13)
    logging_steps = 5,
    optim = 'paged_adamw_8bit',
    lr_scheduler_type = 'cosine',
    seed = 13,
    save_strategy = 'steps',
    save_steps = 25,
    save_total_limit = 3,
    report_to = 'none',            # set 'wandb' after wandb.login()
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LEN,
    packing = False,
    args = args,
)

In [ ]:
# 7b. (Refusal-aware focus, optional but recommended) Train only on the
# assistant's structured response, masking the prompt tokens. This concentrates
# the loss on the decision/decline, per PRD §5.4.
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part = '<|im_start|>assistant\n',
)

In [ ]:
# 8. Train. Auto-resumes from the latest Drive checkpoint if one exists.
import os, glob
ckpts = glob.glob(f'{CKPT_DIR}/{RUN_NAME}/checkpoint-*')
resume = len(ckpts) > 0
print('Resuming from checkpoint' if resume else 'Fresh run')
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# 9. Save the LoRA adapter to Drive.
adapter_path = f'{OUT_DIR}/{RUN_NAME}-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print('Saved adapter ->', adapter_path)

In [ ]:
# 10. Sanity check: does it decline an ill-posed problem and answer a good one?
FastLanguageModel.for_inference(model)
SYSTEM = ('You are a careful point-set topology expert. Solve well-posed problems. '
          'If a problem is FALSE, UNDERSPECIFIED, or OUTSIDE point-set topology, you '
          'must DECLINE rather than guess.')
SCHEMA = ('Respond with ONLY a single JSON object: {"decision":"answer"|"decline",'
          '"confidence":0-1,"reason":"...","answer":"..."}')
tests = [
    'Prove that every compact subset of a topological space is closed.',   # FALSE (needs Hausdorff)
    'Prove that the continuous image of a compact space is compact.',      # answerable
]
for t in tests:
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':f'{t}\n\n{SCHEMA}'}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(ids, max_new_tokens=400, do_sample=False)
    print('Q:', t)
    print('A:', tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    print('-'*70)

## Next: evaluate
Download the adapter (or keep in Drive) and run the eval harness against the held-out test set:
```bash
python -m src.eval.run_eval --backend hf \
  --base Qwen/Qwen2.5-Math-7B-Instruct --adapter out/qwen7b-math-adapter \
  --test data/dataset_test.jsonl --judge-provider openai --out data/eval_finetuned.json
```
Then compare against the frontier baseline with `python -m src.eval.compare`.